# A callable stepped-coupon bond

This notebook values a six-year USD note that amortizes (repays principal in
steps). Its first coupon is fixed. Later coupons float on SOFR, with a cap and
a floor. The issuer can call the note during set windows.

Because the issuer decides whether to call based on future rates, the value
depends on how rates may move. We therefore fit (calibrate) a short-rate model
to market option prices and value the note with two different numerical
methods, then compare them.

The notebook focuses on the contract and its value. Other pieces live in
separate files:

- [the local loading helper](market_data.py) reads the market data;
- [data/bond_tree.json](data/bond_tree.json) and
  [data/bond_fourier.json](data/bond_fourier.json) hold the model and grid
  settings;
- [the scripted cashflows walkthrough](../../scripting/scripted_cashflows.ipynb)
  writes the same note's payments as a script.

See [setup](../../README.md).

In [1]:
import pandas as pd
import quantsupport as qs
from market_data import load_market, load_pricers

market = load_market()
ref = market.quotes.reference_date
tree_configuration = load_pricers("bond_tree.json")
fourier_configuration = load_pricers("bond_fourier.json")
print("Market reference date:", ref)


Market reference date: 2026-02-24


## Write the contract in financial terms

Coupons are described by `FixedRateCoupon`, `FloatingRateCoupon` and
`OptionEmbeddedCoupon`. The issuer's call rights are described by
`CallScheduleEntry`. Dates and conventions are explicit typed values.

The coupon list and the outstanding-principal list are **read by payment
period**: the first element applies to period one, the second to period two,
and the last element repeats for all later periods. So the 4.5% fixed coupon
applies only to the first half-year, not the first year.

Outstanding principal is:

- 100% in period one;
- 90% in period two;
- 75% from period three until maturity.

Each reduction is paid at the end of its period, and the rest is repaid at
maturity. The fixed coupon accrues on an Actual/365 basis. The floating coupon
uses SOFR's Actual/360 convention. It pays SOFR plus 60 bp, floored at 0% and
capped at 6%; the cap and floor apply to the full rate including the spread.

In [2]:
bond = qs.SteppedCouponBond(
    identifier="USD-STEPPED-CALL-6Y",
    start_date=ref,
    maturity_date=ref + "6Y",
    notional=1_000_000,
    units=100.0,
    currency=qs.Currency.USD,
    payment_frequency=qs.Frequency.Semiannual,
    coupon_day_counter=qs.DayCounter.Actual365,
    coupons=[
        qs.FixedRateCoupon(rate=qs.InterestRate(rate=0.045)),
        qs.OptionEmbeddedCoupon(
            index=qs.MarketIndex.SOFR,
            payoff=(qs.PayoffOps.Index + 0.006).max(0.0).min(0.06),
        ),
    ],
    notional_percentages=[1.0, 0.90, 0.75],
    call_schedule=[
        qs.CallScheduleEntry.window(ref + "3Y", ref + "3Y3M", 101.0),
        qs.CallScheduleEntry.window(ref + "4Y", ref + "4Y3M", 100.0),
    ],
)
bond


SteppedCouponBond(business_day_convention=None, calendar=None, call_schedule=[{'end_date': '2029-05-24', 'start_date': '2029-02-24', 'strike': 101.0}, {'end_date': '2030-05-24', 'start_date': '2030-02-24', 'strike': 100.0}], coupon_day_counter='Actual365', coupons=[{'FixedRateCoupon': {'notional': 1.0, 'rate': {'rate': 0.045, 'rate_definition': {'compounding': 'Simple', 'day_counter': 'Actual360', 'frequency': 'Annual'}}}}, {'OptionEmbeddedCoupon': {'day_counter': 'Actual360', 'fixing': None, 'index': 'SOFR', 'notional': 1.0, 'payoff': {'Min': [{'Max': [{'Plus': ['Index', {'Const': 0.006}]}, {'Const': 0.0}]}, {'Const': 0.06}]}, 'spread': 0.0}}], currency='USD', date_generation_rule=None, end_of_month=None, first_coupon_date=None, identifier='USD-STEPPED-CALL-6Y', maturity_date='2032-02-24', next_to_last_coupon_date=None, notional=1000000.0, notional_percentages=[1.0, 0.9, 0.75], payment_frequency='Semiannual', start_date='2026-02-24', termination_date_convention=None, units=100.0)

Each call window includes its first and last day. Call prices (strikes) are
per 100 of outstanding principal. If the issuer calls on a date, the coupon
and amortization due that day are paid first, and the strike is the most the
remaining bond can be worth. The numerical time grid sets which days inside
each window the model can exercise on.

The pricing context's discounting policy discounts USD payments on the SOFR
curve. You pass this policy to the context; it is not part of the contract.

## Model and numerical method

Both configuration files use a Hull–White short-rate model for SOFR with mean
reversion `alpha = 0.05`. Its volatility (sigma) changes over time. Each
segment is fitted to at-the-money (ATM) caplet prices at expiries of 3M, 6M,
1Y, 2Y, 3Y and 5Y. The fit turns the quoted Black volatilities into
absolute short-rate sigmas. The last segment is extended past five years to
cover this six-year note. If you change the inputs, check that the ATM
strikes stay inside the range of strikes in the quoted smiles.

Both methods use 48 time steps per year, up to a maximum of 4,096 steps:

- The tree method works backward through a trinomial tree of rates.
- The Fourier method works backward on a grid of 256 short-rate values that
  spans ten standard deviations on each side.

We compare their values, then rerun with 96 steps per year and, for the
Fourier method, 512 grid points. Before you choose a tolerance for real use,
check that price and risk stay stable under further refinement, including a
wider rate grid.


In [3]:
def price_with(configuration, anchor, requests, instrument=bond):
    context = qs.PricingContext(
        quotes=market.quotes,
        curves=market.curves,
        csa_policy=market.csa_policy,
        volatility_surfaces=market.volatility_surfaces,
        pricer_configurations=configuration,
    )

    with context:
        return context.evaluate(
            qs.Trade(instrument, ref, pricing_anchor=anchor), requests
        )


# 75 bp, held fixed for both engines. Only the tree risk is used below.
anchor = qs.BondPricingAnchor.OAS(0.0075)
tree = price_with(
    tree_configuration,
    anchor,
    [qs.Request.Value, qs.Request.OAS, qs.Request.Sensitivities],
)
fourier = price_with(fourier_configuration, anchor, [qs.Request.Value, qs.Request.OAS])
comparison = pd.DataFrame(
    [
        {"engine": "Tree", "pv_usd": tree.price, "oas_bp": tree.oas * 10_000},
        {
            "engine": "Backward Fourier",
            "pv_usd": fourier.price,
            "oas_bp": fourier.oas * 10_000,
        },
    ]
).set_index("engine")
print(f"Absolute engine difference: USD {abs(tree.price - fourier.price):,.4f}")
comparison


Absolute engine difference: USD 25.4486


,pv_usd,oas_bp
engine,,
Tree,974386.001812,75.0
Backward Fourier,974360.553252,75.0


The table shows the position's PV in USD, with its sign, and the
option-adjusted spread (OAS) we supplied: 75 bp. The OAS is a constant spread
added to the model's discount rates. The two methods differ because they
discretize the problem differently. We compute sensitivities only with the
tree method, for the risk report below.

## Inferring OAS from a price

An OAS pricing anchor values the bond at a given spread. A price anchor does
the reverse: it finds the spread that matches a quoted price. Below we turn
the tree value into a dirty price per 100 and solve for the OAS again. We
should get back 75 bp.

The note settles on its issue date, so the outstanding principal equals the
original notional and there is no accrued interest. For a bond that has
already paid coupons or amortized, use its outstanding principal and accrued
interest at settlement in this conversion.


In [4]:
dirty_quote = tree.price / 1_000_000 * 100
implied = price_with(
    tree_configuration,
    qs.BondPricingAnchor.dirty_price(dirty_quote),
    [qs.Request.Value, qs.Request.OAS],
)
pd.Series(
    {
        "dirty_price_per_100": dirty_quote,
        "implied_oas_bp": implied.oas * 10_000,
        "position_pv_usd": implied.price,
    }
)


dirty_price_per_100        97.438600
implied_oas_bp             75.000000
position_pv_usd        974386.001812
dtype: float64

## Read curve, volatility, and OAS risk separately

The sensitivity table has three kinds of rows:

- the SOFR curve quotes;
- the caplet volatility quotes used to fit the model;
- `<identifier>.OAS`, the sensitivity to the OAS itself.

The curve and volatility sensitivities hold the OAS fixed. They do not show
what happens if the OAS is solved again after each market move to match the
same price.

To report P&L, multiply rate and OAS sensitivities by `1e-4` for a +1 bp move.
Multiply Black volatility sensitivities by `0.01` for a +1 volatility point
move. Yield to maturity and modified duration are not available for this
callable bond. Use the model value, the OAS and the sensitivities instead.

In [5]:
risk = tree.sensitivities.set_index("pillar")["value"]
curve_labels = market.curves[0].to_dict()["quotes"]
vol_labels = market.volatility_surfaces[0].to_dict()["quotes"]
report_groups = [
    ("rate +1 bp", curve_labels, 1e-4),
    ("Black vol +1 point", vol_labels, 0.01),
    ("OAS +1 bp", [bond.identifier + ".OAS"], 1e-4),
]
risk_tables = []
for move, labels, change in report_groups:
    pnl = risk.reindex(labels, fill_value=0) * change
    table = pd.DataFrame({"pnl_usd": pnl, "move": move})
    risk_tables.append(table)
risk_report = pd.concat(risk_tables)
largest_risks = risk_report["pnl_usd"].abs().sort_values(ascending=False).head(12).index
risk_report.loc[largest_risks]


,pnl_usd,move
pillar,,
USD-STEPPED-CALL-6Y.OAS,-328.814193,OAS +1 bp
CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.045_Straddle_Black,-80.936469,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_2Y_Absolute_0.035_Straddle_Black,-80.888198,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_3Y_Absolute_0.035_Straddle_Black,-63.296194,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.045_Straddle_Black,-57.766632,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_3M_Absolute_0.045_Straddle_Black,-56.755505,Black vol +1 point
OIS_USD_SOFR_1Y,-47.276635,rate +1 bp
CapletFloorlet_USD_SOFR_3M_1Y_Absolute_0.035_Straddle_Black,-38.849426,Black vol +1 point
CapletFloorlet_USD_SOFR_3M_6M_Absolute_0.055_Straddle_Black,-19.331666,Black vol +1 point


## Issuer option value and grid refinement

On a call date, the holder's remaining value cannot exceed what the issuer
would pay to redeem the note, after that day's payments. So the same note
without the calls, with the same coupons and model, is worth at least as much
as the callable note.

We value that noncallable note with a continuously compounded Z-spread of
75 bp. Then we value the callable note again on finer grids with both methods.
We divide the USD changes by 10,000 to express them per 100 of the original
USD 1 million principal. A single refinement is not enough to set a
tolerance: also check further refinements, the width of the rate grid and the
risk numbers.


In [6]:
underlying_terms = bond.to_dict()
underlying_terms["call_schedule"] = []
underlying = qs.SteppedCouponBond.from_dict(underlying_terms)
noncallable = price_with(
    tree_configuration,
    qs.BondPricingAnchor.z_spread(0.0075),
    [qs.Request.Value],
    underlying,
)

refined_rows = []
for label, configuration, coarse in [
    ("Tree", tree_configuration, tree),
    ("Backward Fourier", fourier_configuration, fourier),
]:
    specification = configuration[0].to_dict()
    engine = next(iter(specification["pricer"].values()))
    engine["time_grid"]["steps_per_year"] = 96
    if "fourier_grid" in engine:
        engine["fourier_grid"]["points"] = 512
    refined_configuration = [qs.PricerConfiguration.from_dict(specification)]
    refined = price_with(refined_configuration, anchor, [qs.Request.Value])
    refined_rows.append(
        {
            "engine": label,
            "teaching_grid_pv_usd": coarse.price,
            "refined_grid_pv_usd": refined.price,
            "change_per_100_original_principal": (refined.price - coarse.price)
            / 10_000,
        }
    )
print(
    f"Noncallable PV: USD {noncallable.price:,.2f}; callable PV: USD {tree.price:,.2f}."
)
print(f"Issuer call-option cost to holder: USD {noncallable.price - tree.price:,.2f}.")
refinement = pd.DataFrame(refined_rows).set_index("engine")
refinement


Noncallable PV: USD 988,569.06; callable PV: USD 974,386.00.
Issuer call-option cost to holder: USD 14,183.06.


,teaching_grid_pv_usd,refined_grid_pv_usd,change_per_100_original_principal
engine,,,
Tree,974386.001812,974459.894944,0.007389
Backward Fourier,974360.553252,974449.997276,0.008944


The cost of the issuer's call option to the holder is the noncallable value
minus the callable value: USD 14,183.06 here. The OAS round trip checks that
the price and spread are consistent. The grid changes show how stable the
numerical result is for this model.

The volatility sensitivities include the effect of refitting each sigma
segment to its caplet quotes. They do not refit the model when a curve quote
moves.


The [scripted-cashflow notebook](../../scripting/scripted_cashflows.ipynb)
values the noncallable version of this note and checks it at zero volatility.
Use this notebook's callable-bond pricer for the issuer's call.